In [0]:
!pip install openpyxl

In [0]:
import pandas as pd
import glob
import re

In [0]:
def extract_date_from_filename(filename):
    """Extract date from filename in format 'dd.mm.yyyy'."""
    match = re.search(r'(\d{2})\.(\d{2})\.(\d{4})\s*-\s*', filename)
    if match:
        day, month, year = match.groups()
        return f'{year}-{month}-{day}'  # Format as YYYY-MM-DD
    return None

def add_reporting_date(file_paths):
    # Create an empty list to store DataFrames
    dfs = []
    for file in file_paths:
        # Extract the filename
        filename = file.split('/')[-1]
        
        # Read the Excel file
        df = pd.read_excel(file, dtype=str,sheet_name='Sheet1')
        
        # Extract date from filename for all files
        reporting_date = extract_date_from_filename(filename)
        df['Reporting Date'] = reporting_date
        
        dfs.append(df)
    return dfs

In [0]:
# main path
source_path = "/dbfs/mnt/stppeedp/ppeedp/landing/data0/staging/eag/ey/ap_automation/"

# Define the path to write the output 
destination_path = 'dbfs:/mnt/stppeedp/ppeedp/prod/eag/ey/fdw/mfr/manage_engine_vim'
me_vim_paths = source_path + 'manage_engine_vim/*.xlsx'

In [0]:
# Define the path to your Excel files
file_paths = glob.glob(me_vim_paths)

# Add Reporting Date Column
dfs = add_reporting_date(file_paths)

# Concatenate all DataFrames
combined_df = pd.concat(dfs, ignore_index=True)

# Convert to Spark DataFrame
me_vim_df = spark.createDataFrame(combined_df)

# Cast all columns to StringType to avoid error at the time of write
me_vim_df = me_vim_df.select([col(c).cast("string") for c in me_vim_df.columns])

# Write the data to the destination
me_vim_df.write.mode("overwrite").parquet(destination_path)
print(f"Data has been successfully processed and written at {destination_path}")